[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%205/L9_Demo_SteamReviewAssistant.ipynb)

# ISBA 2411 · Week 5 · Lecture 9
## Should You Play It? — a Steam Game-Review Assistant

Last week you *built* a neural network from scratch. Tonight you *borrow* pretrained ones off **Hugging Face** and point them at thousands of real **Steam game reviews** — no training, three lines each.

Tonight's tools:
- **Ask the reviews** — *question answering*: “how hard is Sekiro?” → it extracts the answer from real reviews.
- **Review Router** — *zero-shot classification*: sort any review into categories **you invent on the spot**.
- **Multi-lens work-up** — *sentiment · emotion · irony · toxicity* on any pasted review, all at once.

Then we put them in a live, Steam-styled **chat app** the whole class uses — with cover art, a recommend gauge, a shared *Confidently-Wrong Wall*, and an *analyze a whole game* view — and spend the demo trying to *break* it. The skill isn't calling the model, it's **judging** it.

---
**Runtime:** Colab. Free CPU works; **GPU is much faster** for the Router (Runtime → Change runtime type → GPU).

## 0 · Setup

In [ ]:
%pip install -q "transformers>=4.40,<5" "gradio>=4.0" scikit-learn pandas matplotlib "qrcode[pil]"

import pandas as pd, re, html, textwrap
from transformers import pipeline
print('setup complete')

## 1 · Load the Steam reviews
~5,000 English reviews across 16 games. Each row has the review text, the player's **thumbs-up/down** (`recommended`) and `hours_played`. We also keep each game's Steam **app-id** for cover art.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%205/steam_reviews_demo.csv'

df = pd.read_csv(DATA_URL)
df['review'] = df['review'].map(lambda t: re.sub(r'\s+', ' ', html.unescape(str(t))).strip())
df = df[df['review'].str.len() > 40].reset_index(drop=True)

APPIDS = {'Elden Ring':1245620, "Baldur's Gate 3":1086940, 'Cyberpunk 2077':1091500,
          'Stardew Valley':413150, 'The Witcher 3':292030, 'Counter-Strike 2':730,
          'Terraria':105600, 'Hollow Knight':367520, 'Hades':1145360, 'Portal 2':620,
          'Grand Theft Auto V':271590, 'Red Dead Redemption 2':1174180, 'Sekiro':814380,
          "No Man's Sky":275850, 'DOOM Eternal':782330, 'Team Fortress 2':440}
def cover(game):
    aid = APPIDS.get(game)
    return f'https://cdn.cloudflare.steamstatic.com/steam/apps/{aid}/header.jpg' if aid else None

print(f'{len(df):,} reviews | {df.game.nunique()} games')
df.sample(3)

## 2 · Warm-up — sentiment, and a reality check
You met sentiment in Week 4; here it is the *easy* way — a Hugging Face `pipeline`. But every review carries the player's real **thumbs-up/down**, so we can *grade the model against ground truth.* When the model's call and the player's thumb disagree, that's a miss you can see.

In [ ]:
sentiment = pipeline('sentiment-analysis')

sample = df.sample(40, random_state=0).copy()
sample['model'] = [sentiment(r[:512])[0]['label'] for r in sample['review']]
sample['player'] = sample['recommended'].map({'Recommended':'POSITIVE','Not Recommended':'NEGATIVE'})
agree = (sample['model'] == sample['player']).mean()
print(f'Model agrees with the player thumb {agree:.0%} of the time (n=40)')

print('\n--- a few misses (model vs player) ---')
miss = sample[sample['model'] != sample['player']].head(4)
for r in miss.itertuples():
    print(f"model={r.model:8} player={r.player:8} | {textwrap.shorten(r.review, 90)}")

## 3 · Two models, one review — which do you trust?
Model choice matters. Here the movie-trained **SST-2** model meets a **product-review-trained** model on the same tricky reviews. Where they disagree is where *your judgment* earns its keep.

In [ ]:
stars = pipeline('sentiment-analysis', model='nlptown/bert-base-multilingual-uncased-sentiment')
def star_label(t):
    n = int(stars(t[:512])[0]['label'][0])   # '1 star'..'5 stars'
    return 'POSITIVE' if n >= 4 else 'NEGATIVE' if n <= 2 else 'NEUTRAL'

tricky = ['10/10 would rage quit again.',
          "It's not a bad game.",
          'Great game if you enjoy losing your entire weekend.',
          'I have 400 hours and I hate every second.',
          'Peak. Simply peak.']
print(f"{'review':45} {'SST-2':10} {'review-model'}")
for t in tricky:
    print(f"{textwrap.shorten(t,43):45} {sentiment(t)[0]['label']:10} {star_label(t)}")

## 4 · Tool A — Review Router (zero-shot classification)
**No training, no fixed categories.** You hand the model a review *and labels you invent*, and it scores each one. Under the hood it asks, per label, *“does this review imply it's about ___?”* — the entailment probability is the score. **One pretrained model, infinite label sets.**

In [ ]:
router = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

review = 'Gorgeous game but it ran like garbage on my PC for the first month. Fixed now, worth it.'
labels = ['graphics', 'performance', 'price or value', 'difficulty', 'story']
r = router(review, candidate_labels=labels, multi_label=True)
for lab, sc in zip(r['labels'], r['scores']):
    print(f"{lab:16} {'#'*int(sc*20):20} {sc:.2f}")

### The same review, a different question
Change the labels and you change the *question you're asking the data.* Same text, new lens:

In [ ]:
for lab in [['a complaint','a recommendation','a bug report'],
            ['about the price','about the graphics','about the story']]:
    r = router(review, candidate_labels=lab, multi_label=False)
    print(lab, '->', r['labels'][0], f"({r['scores'][0]:.2f})")

## 5 · Tool B — Ask the reviews (extractive QA)
Given a **question** + **review text** as context, the model *points to the answer span in the text.* It extracts, it doesn't invent — but it can only return words already there.

In [ ]:
qa = pipeline('question-answering', model='distilbert-base-cased-distilled-squad')

context = ('Elden Ring is brutal but fair. The open world is stunning, though the '
           'difficulty spikes and some late bosses feel unfair.')
for q in ['How hard is the game?', 'What do people like about it?']:
    a = qa(question=q, context=context)
    print(f"Q: {q}\n   A: {a['answer']!r}  (score {a['score']:.2f})\n")

### Give QA the *right* reviews to read (Week-4 callback)
We reuse **TF-IDF retrieval** to pull the reviews most relevant to the question — within a game if it names one. Keyword search picks *what to read*; QA does *the reading*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

vec = TfidfVectorizer(stop_words='english', max_features=6000)
M = vec.fit_transform(df['review'])
GAMES = sorted(df['game'].unique())
ALIASES = {'bg3':"Baldur's Gate 3",'baldur':"Baldur's Gate 3",'gta':'Grand Theft Auto V',
           'rdr2':'Red Dead Redemption 2','red dead':'Red Dead Redemption 2','cs2':'Counter-Strike 2',
           'counter strike':'Counter-Strike 2','tf2':'Team Fortress 2','witcher':'The Witcher 3',
           'doom':'DOOM Eternal','stardew':'Stardew Valley','no mans sky':"No Man's Sky"}

def detect_game(text):
    t = text.lower()
    for g in GAMES:
        if g.lower() in t: return g
    for k, v in ALIASES.items():
        if k in t: return v
    return None

def top_reviews(query, k=3, game=None):
    idx = df.index[df['game'] == game] if game else df.index
    sims = linear_kernel(vec.transform([query]), M[idx]).ravel()
    return df.loc[idx[sims.argsort()[::-1][:k]], 'review'].tolist()

print(top_reviews('how hard is it', k=1, game='Sekiro')[0][:200])

## 6 · Three more lenses — emotion · irony · toxicity
The Router isn't the only model we can borrow. Three more `pipeline` calls give a review a full **work-up**: what emotion it carries, whether it's being ironic, and whether it's toxic. Same idea — borrow, don't build — and each has its own blind spots (the irony model misses plenty).

In [ ]:
emotion = pipeline('text-classification', model='j-hartmann/emotion-english-distilroberta-base', top_k=None)
irony   = pipeline('text-classification', model='cardiffnlp/twitter-roberta-base-irony', top_k=None)
toxic   = pipeline('text-classification', model='unitary/toxic-bert', top_k=None)

def _flat(res):
    return res[0] if res and isinstance(res[0], list) else res
def _score(res, label):
    return next((d['score'] for d in _flat(res) if d['label'] == label), 0.0)

r = '10/10 would rage quit again.'
print('emotion:', _flat(emotion(r))[0]['label'],
      '| irony:', round(_score(irony(r), 'irony'), 2),
      '| toxic:', round(_score(toxic(r), 'toxic'), 2))

## 7 · Analyze a whole game — voice of customer
One review is an anecdote; **all** of a game's reviews are a dataset. Run the Router across a game's reviews and average the scores → a topic breakdown. (Batched; first call per game is the slow one, then cached. GPU recommended.)

In [ ]:
TOPIC_LABELS = ['difficulty', 'story', 'graphics', 'price or value', 'bugs or performance', 'multiplayer']
_AGG = {}
def analyze_game(game, n=24):
    if game in _AGG: return _AGG[game]
    revs = df[df['game'] == game]['review'].sample(min(n, (df['game']==game).sum()),
                                                    random_state=0).str.slice(0, 400).tolist()
    res = router(revs, candidate_labels=TOPIC_LABELS, multi_label=True)
    res = res if isinstance(res, list) else [res]
    totals = {l: 0.0 for l in TOPIC_LABELS}
    for one in res:
        for l, s in zip(one['labels'], one['scores']): totals[l] += s
    out = sorted(((l, v/len(res)) for l, v in totals.items()), key=lambda x: -x[1])
    _AGG[game] = out
    return out

import matplotlib.pyplot as plt
g = 'Cyberpunk 2077'
labs, vals = zip(*analyze_game(g))
plt.figure(figsize=(7,3)); plt.barh(labs[::-1], vals[::-1], color='#57b0e6')
plt.title(f'What {g} reviews are about'); plt.xlabel('avg zero-shot score'); plt.tight_layout(); plt.show()

## 8 · Wire it together — styled answer cards
The assistant returns Steam-styled HTML. **Questions** get a QA card (cover art, recommend gauge, answer span, confidence). **Pasted reviews** get the **multi-lens work-up** (sentiment · emotion · irony · toxicity) *plus* the Router tags. Same HTML the chat app renders — it previews right here.

In [ ]:
from IPython.display import HTML, display
import math

QUESTION_STARTS = {'what','why','how','is','are','does','do','can','should','which','when','who','worth'}
DEFAULT_LABELS  = 'difficulty, story, graphics, price or value, bugs or performance, multiplayer'
BAR_COLORS = ['#57b0e6', '#a6cf46', '#c98bdc', '#e0a24e', '#5f7183', '#5f7183']
EMO_EMOJI  = {'anger':'&#128544;','joy':'&#128516;','sadness':'&#128546;','fear':'&#128552;','surprise':'&#128562;','disgust':'&#129314;','neutral':'&#128528;'}
_KEYS = '<style>@keyframes sypgrow{from{width:0 !important}}</style>'
WALL, LAST = [], {}          # WALL is shared across all users of the running app

def looks_like_question(t):
    t = t.strip().lower()
    return t.endswith('?') or (t.split() and t.split()[0] in QUESTION_STARTS)

def _card(inner):
    return (_KEYS + '<div style="background:#151d27;border:1px solid #243040;border-radius:13px;'
            'padding:13px 15px;font-family:system-ui,sans-serif;max-width:560px;">' + inner + '</div>')

def _fill(pct, color):
    return ('<span style="height:7px;background:#0e1620;border:1px solid #243040;border-radius:99px;'
            'display:block;overflow:hidden;">'
            f'<span style="display:block;height:100%;width:{pct}%;background:{color};'
            'animation:sypgrow .7s ease-out;"></span></span>')

def _gauge(pct, color):
    r = 22; c = 2*math.pi*r; off = c*(1-pct/100)
    return (f'<svg width="58" height="58" viewBox="0 0 58 58" style="flex:none;">'
            f'<circle cx="29" cy="29" r="{r}" fill="none" stroke="#22303f" stroke-width="6"/>'
            f'<circle cx="29" cy="29" r="{r}" fill="none" stroke="{color}" stroke-width="6" stroke-linecap="round" '
            f'stroke-dasharray="{c:.1f}" stroke-dashoffset="{off:.1f}" transform="rotate(-90 29 29)"/>'
            f'<text x="29" y="33" text-anchor="middle" font-family="monospace" font-size="13" '
            f'font-weight="700" fill="{color}">{pct}%</text></svg>')

def _tag_rows(pairs):
    rows = ''
    for i, (lab, sc) in enumerate(pairs):
        rows += ('<div style="display:grid;grid-template-columns:150px 1fr 42px;align-items:center;gap:10px;margin:7px 0;">'
                 f'<span style="font-size:13px;color:#dbe4ee;">{lab}</span>'
                 + _fill(int(round(sc*100)), BAR_COLORS[i % len(BAR_COLORS)]) +
                 f'<span style="font-family:monospace;font-size:11.5px;color:#93a4b7;text-align:right;">{sc:.2f}</span></div>')
    return rows

def _label(text, color):
    return (f'<div style="font-family:monospace;font-size:10.5px;letter-spacing:1.2px;'
            f'text-transform:uppercase;color:{color};margin:2px 0 8px;">{text}</div>')

def _chip(name, value, color):
    return ('<span style="display:inline-flex;gap:6px;align-items:center;background:#0f1720;'
            'border:1px solid #22303f;border-radius:8px;padding:5px 9px;margin:0 6px 6px 0;font-size:12px;">'
            f'<span style="color:#7a8b9a;">{name}</span>'
            f'<span style="color:{color};font-weight:600;">{value}</span></span>')

def _lens_html(text):
    t = text[:512]
    s = sentiment(t)[0]; scol = '#a6cf46' if s['label'] == 'POSITIVE' else '#e5705f'
    emo = _flat(emotion(t))[0]
    iro = _score(irony(t), 'irony'); tox = _score(toxic(t), 'toxic')
    chips = (_chip('sentiment', s['label'].title(), scol)
             + _chip('emotion', EMO_EMOJI.get(emo['label'], '') + ' ' + emo['label'], '#c98bdc')
             + _chip('irony', ('&#128681; %d%%' % round(iro*100)) if iro > 0.5 else ('%d%%' % round(iro*100)),
                     '#e0a24e' if iro > 0.5 else '#7a8b9a')
             + _chip('toxicity', ('&#128681; %d%%' % round(tox*100)) if tox > 0.5 else 'clean &#10003;',
                     '#e5705f' if tox > 0.5 else '#a6cf46'))
    return _label('Multi-lens &middot; four models, one review', '#57b0e6') + '<div>' + chips + '</div>'

def _qa_html(game, answer, score):
    verdict = ''
    if game is not None:
        g = df[df['game'] == game]
        pct = int(round((g['recommended'] == 'Recommended').mean() * 100))
        gcol = '#a6cf46' if pct >= 60 else '#e0a24e' if pct >= 45 else '#e5705f'
        art = (f'<img src="{cover(game)}" alt="" style="width:120px;border-radius:8px;'
               'border:1px solid #2d3a4c;flex:none;" onerror="this.style.display=&quot;none&quot;">'
               if cover(game) else '')
        verdict = ('<div style="display:flex;align-items:center;gap:12px;padding:10px;border-radius:10px;'
                   'background:#0f1720;border:1px solid #22303f;margin-bottom:11px;">'
                   + art + _gauge(pct, gcol) +
                   f'<span style="font-size:12.5px;color:#c8d3df;"><b style="color:#eaf2fb;">{game}</b><br>'
                   f'{pct}% of {len(g)} reviewers recommend it</span></div>')
    ans = ('<div style="font-size:13.5px;color:#dbe4ee;line-height:1.5;">From the reviews: '
           f'<mark style="background:#123049;color:#bfe2fb;padding:1px 6px;border-radius:5px;">{answer}</mark></div>')
    meter = ('<div style="display:flex;align-items:center;gap:9px;margin-top:11px;">'
             '<span style="font-size:11.5px;color:#93a4b7;">answer confidence</span>'
             + _fill(int(round(score*100)), '#57b0e6') +
             f'<span style="font-family:monospace;font-size:11.5px;color:#93a4b7;">{score:.2f}</span></div>')
    return _card(_label('Ask the reviews', '#57b0e6') + verdict + ans + meter)

def assistant(message, history=None, labels=DEFAULT_LABELS):
    msg = message.strip()
    if msg.lower().startswith('analyze '):
        game = detect_game(msg[8:]) or detect_game(msg)
        if game:
            return _card(_label(f'What {game} reviews are about', '#a6cf46') + _tag_rows(analyze_game(game)))
        return _card('<span style="color:#e5705f">Name a game I have, e.g. <b>analyze Elden Ring</b>.</span>')
    if looks_like_question(msg):
        game = detect_game(msg)
        ctx = ' '.join(top_reviews(msg, k=3, game=game))[:2000]
        a = qa(question=msg, context=ctx)
        return _qa_html(game, a['answer'], a['score'])
    labs = [l.strip() for l in (labels or DEFAULT_LABELS).split(',') if l.strip()]
    r = router(msg[:400], candidate_labels=labs, multi_label=True)
    tags = _label('Review Router &middot; zero-shot tags', '#a6cf46') + _tag_rows(list(zip(r['labels'], r['scores']))[:5])
    return _card(_lens_html(msg) + '<div style="height:8px"></div>' + tags)

display(HTML(assistant('Is Elden Ring worth it?')))
display(HTML(assistant('Great game if you enjoy losing your entire weekend.')))

## 9 · The chat app — the whole class joins
Dark Steam-styled chat. `launch(share=True)` prints a **public URL** — put it (and a QR from the next cell) on screen. Ask about a game (QA + gauge + cover art), paste a review (multi-lens + Router tags), type **analyze <game>** for the voice-of-customer view, hit the **trap** buttons for guaranteed sarcasm, and **flag** any confident-wrong answer to the shared **Wall**.

In [ ]:
import gradio as gr

QUICKPICKS = ['Elden Ring', 'Sekiro', 'Cyberpunk 2077', 'Hades', "Baldur's Gate 3", 'Stardew Valley']
TRAPS = ['10/10 would rage quit again.', "It's not a bad game.",
         'Great game if you enjoy losing your entire weekend.']

CSS = """
.gradio-container{background:#0c1118 !important;}
footer{display:none !important;}
#syp-header{display:flex;align-items:center;gap:14px;padding:12px 4px 8px;}
#syp-header .mark{width:38px;height:38px;border-radius:10px;background:#0e1a26;border:1px solid #2d3a4c;
  display:grid;place-items:center;color:#57b0e6;font-size:20px;}
#syp-header .ttl{font-size:17px;font-weight:650;color:#dbe4ee;}
#syp-header .sub{font-size:12px;color:#93a4b7;}
#syp-header .live{margin-left:auto;font-family:ui-monospace,monospace;font-size:11.5px;color:#93a4b7;
  background:#0e1620;border:1px solid #243040;padding:6px 11px;border-radius:999px;}
"""

theme = gr.themes.Base(primary_hue='blue', neutral_hue='slate').set(
    body_background_fill='#0c1118', block_background_fill='#151d27',
    block_border_color='#243040', body_text_color='#dbe4ee',
    input_background_fill='#0d141d', input_border_color='#2d3a4c',
    button_primary_background_fill='#57b0e6', button_primary_text_color='#052033',
)

HEADER = ('<div id="syp-header"><div class="mark">&#127918;</div>'
          '<div><div class="ttl">Should You Play It?</div>'
          '<div class="sub">Steam review assistant &middot; ISBA 2411</div></div>'
          f'<div class="live">&#9679; live &middot; {len(df):,} reviews &middot; {df.game.nunique()} games</div></div>')

def render_wall():
    if not WALL:
        return '<div style="color:#5f7183;font-size:13px;">No flagged answers yet — go break the bot.</div>'
    items = ''.join(
        f'<div style="border:1px solid #3a1712;background:#1c1210;border-radius:9px;padding:9px 11px;margin:6px 0;">'
        f'<div style="color:#e5705f;font-size:12px;font-family:monospace;">#{i+1}</div>'
        f'<div style="color:#dbe4ee;font-size:13px;">{w["user"]}</div></div>'
        for i, w in enumerate(WALL[-12:]))
    return f'<div style="font-family:monospace;font-size:11px;color:#e5705f;margin-bottom:6px;">CONFIDENTLY-WRONG WALL &middot; {len(WALL)}</div>' + items

def respond(message, chat_history, labels_val):
    if not message.strip():
        yield '', chat_history; return
    ch = (chat_history or []) + [{'role': 'user', 'content': message}]
    yield '', ch + [{'role': 'assistant', 'content': '<span style="color:#93a4b7">…reading the reviews</span>'}]
    reply = assistant(message, labels=labels_val)
    LAST['user'] = message; LAST['reply'] = reply
    yield '', ch + [{'role': 'assistant', 'content': reply}]

def flag():
    if LAST.get('user'): WALL.append(dict(LAST))
    return render_wall()

with gr.Blocks(theme=theme, css=CSS, title='Should You Play It?') as demo:
    gr.HTML(HEADER)
    chat = gr.Chatbot(height=430, sanitize_html=False, type='messages', show_label=False)
    with gr.Row():
        picks = [gr.Button(g, size='sm') for g in QUICKPICKS]
    with gr.Row():
        traps = [gr.Button('😈 ' + t[:22] + '…', size='sm') for t in TRAPS]
    with gr.Row():
        msg  = gr.Textbox(placeholder='Ask about a game, paste a review, or "analyze <game>"...',
                          show_label=False, scale=5)
        send = gr.Button('Send', variant='primary', scale=1)
    labels = gr.Textbox(value=DEFAULT_LABELS, label='Router labels (invent your own!)')
    with gr.Row():
        flag_btn = gr.Button('🚩 Flag last answer as confidently wrong', size='sm')
        refresh  = gr.Button('Refresh wall', size='sm')
    wall = gr.HTML(render_wall())

    send.click(respond, [msg, chat, labels], [msg, chat])
    msg.submit(respond, [msg, chat, labels], [msg, chat])
    for b, g in zip(picks, QUICKPICKS):
        b.click(lambda g=g: f'Is {g} worth it?', None, msg)
    for b, t in zip(traps, TRAPS):
        b.click(lambda t=t: t, None, msg)
    flag_btn.click(flag, None, wall)
    refresh.click(render_wall, None, wall)

demo.launch(share=True)

## 10 · Put a QR on the screen
Paste the **public URL** the cell above printed (the `https://….gradio.live` one) and run this to show a QR the room can scan.

In [ ]:
import qrcode
SHARE_URL = 'https://your-link.gradio.live'   # <-- paste the printed public URL
qrcode.make(SHARE_URL)

### Fallback (if campus wi-fi blocks the public link)
Same brain, inline — no public URL.

In [ ]:
import ipywidgets as w
from IPython.display import display, HTML
box = w.Text(placeholder='Ask about a game, or paste a review...', layout=w.Layout(width='70%'))
go = w.Button(description='Send', button_style='primary'); out = w.Output()
def handle(_):
    if not box.value.strip(): return
    ans = assistant(box.value)
    with out: display(HTML(f'<div style="color:#93a4b7;font-size:12px;margin-top:8px;">you: '
                           f'{box.value}</div>' + ans))
    box.value = ''
go.on_click(handle)
display(w.HBox([box, go]), out)

---
## Take-home exercise
1. **Router:** pick 15 reviews and a label set of your own. Find two the Router gets wrong — name why.
2. **Multi-lens:** find a review where the four lenses *disagree* (e.g. sentiment POSITIVE but irony flagged). Which lens do you trust, and why?
3. **QA:** ask five game questions. Find one it nails and one it fumbles — name the failure mode.
4. **Aggregate:** `analyze` two games. Does the topic mix match the game's reputation?

Hand in a short write-up: approach, findings, three failure cases. *(Maps to the midterm rubric's Analysis & Insight.)*